# SA-DMAE vs DMAE Baseline 비교

**비교 전략:** 가장 공정한 비교를 위해 동일 데이터로 DMAE 베이스라인을 직접 학습

| | DMAE (Baseline) | SA-DMAE (Ours) |
|--|--|--|
| 입력 | center slice 1장 | 3장 연속 슬라이스 |
| Axial Attention | ✗ | ✓ (axial_depth=4) |
| 설정 | `n_slices=1, axial_depth=0` | `n_slices=3, axial_depth=4` |

**순서:**
1. DMAE 베이스라인 학습 (Cell 1~3)
2. 두 모델 복원 비교 시각화 (Cell 4~6)
3. 정량 지표 비교 표 (Cell 7)

In [ ]:
# ── Cell 1: 환경 설정 ────────────────────────────────────────────────────────
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/SA-DMAE'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/whkim4338/SA-DMAE.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install timm nibabel tensorboard tqdm -q

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── Cell 2: 경로 설정 (여기만 수정) ─────────────────────────────────────────

BRATS_PT       = '/content/drive/MyDrive/SA_DMAE_slices/brats'
UCSF_PT        = '/content/drive/MyDrive/SA_DMAE_slices/ucsf'

# SA-DMAE 체크포인트 (이미 학습됨)
SA_DMAE_CKPT   = '/content/drive/MyDrive/SA_DMAE_output/checkpoint-best.pth'

# DMAE 베이스라인 출력 경로 (새로 학습)
DMAE_OUT       = '/content/drive/MyDrive/DMAE_baseline_output'
DMAE_CKPT      = f'{DMAE_OUT}/checkpoint-best.pth'

# 비교 Figure 저장 경로
COMPARE_DIR    = '/content/drive/MyDrive/SA_DMAE_output/comparison'

# 베이스라인 학습 설정
EPOCHS         = 200
BATCH_SIZE     = 32
ACCUM_ITER     = 2
PATIENCE       = 30
SAVE_EVERY     = 10

import os
os.makedirs(DMAE_OUT, exist_ok=True)
os.makedirs(COMPARE_DIR, exist_ok=True)
print('경로 설정 완료')

In [ ]:
# ── Cell 3: DMAE 베이스라인 학습 ─────────────────────────────────────────────
# n_slices=1, axial_depth=0 → center slice만 사용, Axial Attention 없음
# 이미 학습된 체크포인트가 있으면 이 셀은 스킵해도 됨

import os
if os.path.exists(DMAE_CKPT):
    print(f'이미 학습된 DMAE 베이스라인 발견: {DMAE_CKPT}')
    print('Cell 4로 바로 이동하세요.')
else:
    print('DMAE 베이스라인 학습 시작...')
    !python main_pretrain_sa.py \
        --data_path  {BRATS_PT} \
        --ucsf_path  {UCSF_PT} \
        --preprocessed \
        --device     cuda \
        --model      sa_dmae_vit_base_patch16 \
        --n_slices   1 \
        --axial_depth 0 \
        --epochs     {EPOCHS} \
        --batch_size {BATCH_SIZE} \
        --accum_iter {ACCUM_ITER} \
        --sigma      0.25 \
        --mask_ratio 0.75 \
        --val_ratio  0.1 \
        --patience   {PATIENCE} \
        --save_every {SAVE_EVERY} \
        --output_dir {DMAE_OUT} \
        --log_dir    {DMAE_OUT} \
        --num_workers 2

In [ ]:
# ── Cell 4: 두 모델 로드 ─────────────────────────────────────────────────────
import torch, sys
sys.path.insert(0, '/content/SA-DMAE')
import models_sa_dmae

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# SA-DMAE 로드
sa_dmae = models_sa_dmae.sa_dmae_vit_base_patch16(
    sigma=0.25, n_slices=3, axial_depth=4
)
ckpt_sa = torch.load(SA_DMAE_CKPT, map_location='cpu', weights_only=False)
sa_dmae.load_state_dict(ckpt_sa['model'])
sa_dmae.to(device).eval()
print(f'SA-DMAE 로드 완료 (epoch {ckpt_sa["epoch"]})')

# DMAE 베이스라인 로드
dmae = models_sa_dmae.sa_dmae_vit_base_patch16(
    sigma=0.25, n_slices=1, axial_depth=0
)
ckpt_dm = torch.load(DMAE_CKPT, map_location='cpu', weights_only=False)
dmae.load_state_dict(ckpt_dm['model'])
dmae.to(device).eval()
print(f'DMAE 베이스라인 로드 완료 (epoch {ckpt_dm["epoch"]})')

In [ ]:
# ── Cell 5: 비교 유틸 함수 ───────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).reshape(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).reshape(3, 1, 1)
MOD_NAMES     = ['T1ce', 'T2', 'FLAIR']
PATCH_SIZE    = 16


def make_masked_image(original, mask, patch_size=16, mask_value=0.5):
    C, H, W = original.shape
    n = H // patch_size
    masked = original.clone()
    mask_2d = mask.reshape(n, n)
    for i in range(n):
        for j in range(n):
            if mask_2d[i, j] == 1:
                masked[:, i*patch_size:(i+1)*patch_size,
                           j*patch_size:(j+1)*patch_size] = mask_value
    return masked


def run_model(model, sample, mask_ratio, device, seed=42):
    """sample: (n_slices, C, H, W) → original, masked, recon 반환"""
    torch.manual_seed(seed)
    x = sample.unsqueeze(0).to(device)
    with torch.no_grad():
        loss, pred, mask = model(x, mask_ratio=mask_ratio)

    recon = model.unpatchify(pred)[0].cpu()
    recon = (recon * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)

    center_idx = sample.shape[0] // 2
    original   = sample[center_idx].cpu()
    masked_img = make_masked_image(original, mask[0].cpu())
    return original, masked_img, recon, loss.item()


def compute_psnr(original, recon):
    """채널 평균 PSNR (dB)"""
    mse  = np.mean((original.numpy() - recon.numpy()) ** 2)
    return 10 * np.log10(1.0 / (mse + 1e-8))


def compute_ssim(original, recon):
    """채널 평균 SSIM (간이 구현)"""
    from skimage.metrics import structural_similarity as ssim
    vals = []
    for c in range(original.shape[0]):
        o = original[c].numpy()
        r = recon[c].numpy()
        vals.append(ssim(o, r, data_range=1.0))
    return float(np.mean(vals))


print('비교 유틸 준비 완료')

In [ ]:
# ── Cell 6: 시각 비교 (논문용 Figure) ────────────────────────────────────────
import random
from pathlib import Path

SEED       = 42
MASK_RATIO = 0.75
N_SAMPLES  = 4   # BraTS 2 + UCSF 2

random.seed(SEED)
torch.manual_seed(SEED)

brats_files = sorted(Path(BRATS_PT).glob('*.pt'))
ucsf_files  = sorted(Path(UCSF_PT).glob('*.pt'))
picked      = random.sample(brats_files, 2) + random.sample(ucsf_files, 2)
src_labels  = ['BraTS', 'BraTS', 'UCSF', 'UCSF']

# ── 모달리티별로 한 장씩 골라 시각화 (T1ce 기준) ────────────────────────────
MOD_IDX = 0   # 0=T1ce, 1=T2, 2=FLAIR
MOD_NAME = MOD_NAMES[MOD_IDX]

n_rows = len(picked)
# 열: Original | Masked | DMAE recon | SA-DMAE recon | Error(DMAE) | Error(SA-DMAE)
n_cols = 6
col_titles = [
    'Original', 'Masked\n(75%)',
    'DMAE\n(Baseline)', 'SA-DMAE\n(Ours)',
    'Error\nDMAE', 'Error\nSA-DMAE'
]

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4.5 * n_rows))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle(
    f'SA-DMAE vs DMAE Baseline — Reconstruction Comparison ({MOD_NAME})\n'
    f'mask ratio = {MASK_RATIO}',
    fontsize=14, fontweight='bold', color='white', y=1.01
)

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=10, color='#CCCCCC',
                           fontweight='bold', pad=5)

all_metrics = []

for row, (pt_path, src) in enumerate(zip(picked, src_labels)):
    sample_3 = torch.load(pt_path, map_location='cpu', weights_only=True)  # (3,3,224,224)
    sample_1 = sample_3[1:2]   # center slice only → (1, 3, 224, 224) for DMAE
    case_id  = pt_path.stem

    # 두 모델 추론 (같은 seed로 동일한 마스크 사용)
    orig_sa, masked_sa, recon_sa, loss_sa = run_model(sa_dmae, sample_3, MASK_RATIO, device, SEED)
    orig_dm, masked_dm, recon_dm, loss_dm = run_model(dmae,    sample_1, MASK_RATIO, device, SEED)

    psnr_sa = compute_psnr(orig_sa, recon_sa)
    psnr_dm = compute_psnr(orig_dm, recon_dm)
    all_metrics.append((src, case_id, loss_dm, loss_sa, psnr_dm, psnr_sa))

    err_dm = np.abs(orig_dm[MOD_IDX].numpy() - recon_dm[MOD_IDX].numpy())
    err_sa = np.abs(orig_sa[MOD_IDX].numpy() - recon_sa[MOD_IDX].numpy())

    imgs  = [
        orig_sa[MOD_IDX].numpy(),
        masked_sa[MOD_IDX].numpy(),
        recon_dm[MOD_IDX].numpy(),
        recon_sa[MOD_IDX].numpy(),
        err_dm, err_sa
    ]
    cmaps = ['gray','gray','gray','gray','hot','hot']
    vmaxs = [1, 1, 1, 1, err_dm.max()+1e-6, err_sa.max()+1e-6]

    for col, (img, cmap, vmax) in enumerate(zip(imgs, cmaps, vmaxs)):
        ax = axes[row, col]
        ax.set_facecolor('black')
        ax.imshow(img, cmap=cmap, vmin=0, vmax=vmax)
        ax.axis('off')

        # PSNR 표시 (복원 열)
        if col == 2:
            ax.text(0.03, 0.04, f'PSNR: {psnr_dm:.1f} dB',
                    transform=ax.transAxes, fontsize=8.5,
                    color='#FFAA00', va='bottom',
                    bbox=dict(fc='black', alpha=0.6, pad=2))
        if col == 3:
            color = '#44FF88' if psnr_sa > psnr_dm else '#FF6644'
            ax.text(0.03, 0.04, f'PSNR: {psnr_sa:.1f} dB',
                    transform=ax.transAxes, fontsize=8.5,
                    color=color, va='bottom',
                    bbox=dict(fc='black', alpha=0.6, pad=2))

    # 행 레이블
    axes[row, 0].set_ylabel(
        f'[{src}]\n{case_id[:18]}',
        fontsize=8, color='white', labelpad=4,
        rotation=0, ha='right', va='center'
    )

plt.tight_layout()
fig_path = f'{COMPARE_DIR}/comparison_{MOD_NAME}.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f'Figure 저장: {fig_path}')

In [ ]:
# ── Cell 7: 전체 테스트셋 정량 비교 ─────────────────────────────────────────
from pathlib import Path
import numpy as np, torch
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim

MASK_RATIO = 0.75
SEED       = 42
N_EVAL     = 100   # 평가에 사용할 샘플 수 (전체 다 하려면 None)

all_files = sorted(list(Path(BRATS_PT).glob('*.pt')) +
                   list(Path(UCSF_PT).glob('*.pt')))
if N_EVAL:
    import random; random.seed(SEED)
    all_files = random.sample(all_files, min(N_EVAL, len(all_files)))

dm_mse_list, dm_psnr_list, dm_ssim_list = [], [], []
sa_mse_list, sa_psnr_list, sa_ssim_list = [], [], []

for pt_path in tqdm(all_files, desc='Evaluating'):
    sample_3 = torch.load(pt_path, map_location='cpu', weights_only=True)
    sample_1 = sample_3[1:2]

    orig_sa, _, recon_sa, _ = run_model(sa_dmae, sample_3, MASK_RATIO, device, SEED)
    orig_dm, _, recon_dm, _ = run_model(dmae,    sample_1, MASK_RATIO, device, SEED)

    for c in range(3):
        o_sa = orig_sa[c].numpy(); r_sa = recon_sa[c].numpy()
        o_dm = orig_dm[c].numpy(); r_dm = recon_dm[c].numpy()

        mse_sa = np.mean((o_sa - r_sa)**2); mse_dm = np.mean((o_dm - r_dm)**2)
        sa_mse_list.append(mse_sa); dm_mse_list.append(mse_dm)
        sa_psnr_list.append(10*np.log10(1/(mse_sa+1e-8)))
        dm_psnr_list.append(10*np.log10(1/(mse_dm+1e-8)))
        sa_ssim_list.append(ssim(o_sa, r_sa, data_range=1.0))
        dm_ssim_list.append(ssim(o_dm, r_dm, data_range=1.0))

print('\n' + '='*52)
print(f'{"":<12} {"MSE":>10} {"PSNR (dB)":>12} {"SSIM":>10}')
print('-'*52)
print(f'{"DMAE":12} {np.mean(dm_mse_list):>10.4f} '
      f'{np.mean(dm_psnr_list):>12.2f} {np.mean(dm_ssim_list):>10.4f}')
print(f'{"SA-DMAE":12} {np.mean(sa_mse_list):>10.4f} '
      f'{np.mean(sa_psnr_list):>12.2f} {np.mean(sa_ssim_list):>10.4f}')
print('='*52)

delta_mse  = np.mean(dm_mse_list)  - np.mean(sa_mse_list)
delta_psnr = np.mean(sa_psnr_list) - np.mean(dm_psnr_list)
delta_ssim = np.mean(sa_ssim_list) - np.mean(dm_ssim_list)
print(f'\nSA-DMAE 개선량:')
print(f'  MSE  ↓ {delta_mse:+.4f}  ({delta_mse/np.mean(dm_mse_list)*100:+.1f}%)')
print(f'  PSNR ↑ {delta_psnr:+.2f} dB')
print(f'  SSIM ↑ {delta_ssim:+.4f}')

# 결과 저장
import json
result = {
    'n_samples': len(all_files),
    'dmae':    {'mse': float(np.mean(dm_mse_list)),
                'psnr': float(np.mean(dm_psnr_list)),
                'ssim': float(np.mean(dm_ssim_list))},
    'sa_dmae': {'mse': float(np.mean(sa_mse_list)),
                'psnr': float(np.mean(sa_psnr_list)),
                'ssim': float(np.mean(sa_ssim_list))},
}
with open(f'{COMPARE_DIR}/metrics.json', 'w') as f:
    json.dump(result, f, indent=2)
print(f'\n지표 저장: {COMPARE_DIR}/metrics.json')